#  Building Resilient AWS OpenSearch Clusters with Cross-Cluster Replication

## Import Required Libraries

Import all the necessary Python libraries for the OpenSearch replication and backup demonstration:

- `boto3`: AWS SDK for Python to interact with AWS services
- `requests`: HTTP library for making API calls to OpenSearch
- `requests_aws4auth`: For AWS Signature Version 4 authentication with OpenSearch
- `os`: For environment variable access
- `dotenv`: To load environment variables from .env file
- `json`: For JSON data handling
- `tabulate`: For displaying data in table format

In [91]:
import os
import json
import boto3
import requests
from requests_aws4auth import AWS4Auth
from dotenv import load_dotenv
from tabulate import tabulate
from datetime import datetime, timezone


## Set Up AWS Authentication

This cell configures AWS authentication for both regions:

- Retrieves AWS credentials from the environment (IAM roles, ~/.aws/credentials, or environment variables)
- Creates two AWS4Auth objects:
  - `auth_south`: For ap-south-1 region (source cluster) 
  - `auth_north`: For us-east-1 region (destination cluster)
- Uses the 'es' service name (required for OpenSearch, even though it's not Elasticsearch)
- Includes session token for temporary credentials (common with IAM roles)

In [92]:
credentials = boto3.Session().get_credentials()
auth_south = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    "ap-south-1",
    "es",
    session_token=credentials.token
)

# 3. Create Auth for North (Source)
auth_north = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    "us-east-1",
    "es",
    session_token=credentials.token
)

## Load Environment Variables into Python Variables

This cell reads the environment variables loaded by `load_dotenv()` and assigns them to Python variables for easier access throughout the notebook:

- `source_host`: Endpoint URL for the source OpenSearch cluster
- `destination_host`: Endpoint URL for the destination OpenSearch cluster
- `source_region`: AWS region where source cluster is located
- `destination_region`: AWS region where destination cluster is located
- `role_arn`: IAM role ARN that has permissions to access S3 bucket
- `bucket_name`: Name of the S3 bucket for storing snapshots
- `repository_name`: Name of the snapshot repository in OpenSearch
- `snapshot_name`: Name of the snapshot to be created

In [93]:
load_dotenv()
source_host = os.getenv('SOURCE_ENDPOINT')
destination_host = os.getenv('DESTINATION_ENDPOINT')
source_region = os.getenv('SOURCE_REGION')
destination_region = os.getenv('DESTINATION_REGION')
role_arn = os.getenv('ROLE_ARN')
bucket_name = os.getenv('BUCKET_NAME')
repository_name = os.getenv('REPOSITORY_NAME')


In [ ]:
#os.environ.clear()

In [ ]:
def delete_all_opensearch_data(host, auth):
    """
    Deletes all indices from an OpenSearch cluster, removing all data.
    """
    url = f"https://{host}/_all"
    headers = {"Content-Type": "application/json"}

    try:
        response = requests.delete(url, auth=auth, headers=headers)
        response.raise_for_status()
        print(f"Successfully deleted all data from {host}")
        return response.json()
    
    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err.response.status_code}")
        print(err.response.text)
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
delete_all_opensearch_data(source_host, auth_south)

In [ ]:
delete_all_opensearch_data(destination_host, auth_north)

## Populate Source OpenSearch Cluster

This cell creates and indexes sample log documents into the source OpenSearch cluster:

- Defines a list of log entries with sample data
- Each log entry contains:
  - `id`: Document ID for indexing
  - `body`: The actual document content with timestamp, level, message, service, and region
- Uses `requests.post()` with AWS4Auth to index documents
- Indexes documents to the `logs` index with specific document IDs
- Prints the HTTP status code for each indexing operation (should be 201 for success)

**Purpose**: Creates test data in the source cluster that will later be replicated and backed up.

In [94]:
logs = [
    {
        "id": "1",
        "body": {
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "level": "INFO",
            "message": "US-East Primary cluster populated",
            "service": "opensearch",
            "region": "us-east-1"
        }
    },
    {
        "id": "2",
        "body": {
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "level": "INFO",
            "message": "Source data ready for CCR to AP-South",
            "service": "opensearch",
            "region": "us-east-1"
        }
    }
]

for log in logs:
    url = f"https://{source_host}/logs/_doc/{log['id']}"
    response = requests.post(url, auth=auth_south, json=log['body'])
    print(f"Doc {log['id']} Status: {response.status_code}")

Doc 1 Status: 201
Doc 2 Status: 201


## View Populated Data from Source Cluster

This cell queries the source OpenSearch cluster and displays the indexed documents in a formatted table:

- Constructs a search URL for the `logs` index
- Uses `requests.get()` with AWS4Auth to perform the search
- Parses the JSON response from OpenSearch
- Extracts document data (ID, timestamp, level, message, region) from search hits
- Uses `tabulate` library to display results in a grid table format
- Handles errors gracefully with try/except block

**Purpose**: Verifies that the documents were successfully indexed and displays them in a readable format.

In [95]:

endpoint = os.getenv('SOURCE_ENDPOINT')
url = f"https://{endpoint}/logs/_search"
params = {"size": 10}

try:
    # 1. Execute the signed request
    response = requests.get(url, auth=auth_south, params=params)
    response.raise_for_status()
    
    # 2. Parse the JSON response
    results = response.json()
    hits = results.get('hits', {}).get('hits', [])

    # 3. Extract the data for the table
    table_data = []
    for hit in hits:
        source = hit.get('_source', {})
        table_data.append([
            hit.get('_id'),
            source.get('timestamp'),
            source.get('level'),
            source.get('message'),
            source.get('region')
        ])

    # 4. Display as a table
    headers = ["ID", "Timestamp", "Level", "Message", "Region"]
    print(tabulate(table_data, headers=headers, tablefmt="grid"))

except Exception as e:
    print(f"Error: {e}")


+------+---------------------------+---------+---------------------------------------+-----------+
|   ID | Timestamp                 | Level   | Message                               | Region    |
+======+===========================+=========+=======================================+===========+
|    1 | 2026-04-04T05:47:21+00:00 | INFO    | US-East Primary cluster populated     | us-east-1 |
+------+---------------------------+---------+---------------------------------------+-----------+
|    2 | 2026-04-04T05:47:21+00:00 | INFO    | Source data ready for CCR to AP-South | us-east-1 |
+------+---------------------------+---------+---------------------------------------+-----------+


## Register S3 Repository Function

This cell defines a reusable function to register an S3 bucket as a snapshot repository in OpenSearch:

**Function Parameters:**
- `host`: OpenSearch cluster endpoint
- `region`: AWS region where the cluster is located
- `repository_name`: Name for the repository (used in snapshot operations)
- `bucket_name`: Name of the S3 bucket to use for snapshots
- `role_arn`: IAM role ARN with S3 permissions
- `auth`: AWS4Auth object for authentication

**Function Process:**
1. Constructs the snapshot repository registration URL
2. Creates payload with S3 repository settings
3. Makes authenticated PUT request to register the repository
4. Returns success confirmation or error details

**Purpose**: Enables OpenSearch to store snapshots in the specified S3 bucket for backup/restore operations.

In [ ]:
def register_opensearch_repository(host, region, repository_name, bucket_name, role_arn, auth):
    """
    Registers an S3 bucket as a snapshot repository in Amazon OpenSearch.
    """
    service = 'es' 
    url = f"https://{host}/_snapshot/{repository_name}"
    payload = {
        "type": "s3",
        "settings": {
            "bucket": bucket_name,
            "region": region,
            "role_arn": role_arn
        }
    }
    headers = {"Content-Type": "application/json"}

    # 4. Execute the request
    try:
        response = requests.put(url, auth=auth, json=payload, headers=headers)
        response.raise_for_status()
        print(f"Successfully registered repository: {repository_name}")
        return response.json()
    
    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


## Register Repository on Source Cluster

This cell calls the repository registration function for the source OpenSearch cluster:

- Registers the S3 bucket as a snapshot repository on the source cluster (ap-south-1)
- Uses `auth_south` for authentication
- Enables the source cluster to create snapshots in the S3 bucket

**Prerequisites**: S3 bucket must exist and the IAM role must have appropriate permissions.

In [96]:
register_opensearch_repository(source_host, source_region, repository_name, bucket_name, role_arn, auth_south)

Successfully registered repository: backup-repo


{'acknowledged': True}

## Register Repository on Destination Cluster


In [97]:
register_opensearch_repository(destination_host, destination_region, repository_name, bucket_name, role_arn, auth_north)

Successfully registered repository: backup-repo


{'acknowledged': True}

## Create Snapshot from Source Cluster

This cell initiates a snapshot creation on the source OpenSearch cluster:

- Constructs the snapshot URL using repository and snapshot names
- Uses `wait_for_completion=true` to wait for the snapshot to finish
- Makes authenticated PUT request to start the snapshot process
- Prints detailed snapshot information including status, indices, and timing

**Process**: 
1. OpenSearch creates a snapshot of all indices in the S3 bucket
2. The snapshot includes index metadata, mappings, and documents
3. Waits for completion before proceeding

**Purpose**: Creates a backup of the source cluster data that can be restored on the destination cluster.

**Note**: By default, this snapshots ALL indices. To snapshot specific indices, modify the payload to include an "indices" parameter.

In [98]:
snapshot_name = f"snapshot_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

url = f"https://{source_host}/_snapshot/{repository_name}/{snapshot_name}?wait_for_completion=true"

print(f"Starting snapshot: {snapshot_name}...")

try:
    response = requests.put(url, auth=auth_south)
    response.raise_for_status()
    print("Success! Snapshot details:")
    print(json.dumps(response.json(), indent=2))
except requests.exceptions.HTTPError as err:
    print(f"Error: {err.response.status_code}")
    print(err.response.text)
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Starting snapshot: snapshot_20260404_112031...
Success! Snapshot details:
{
  "snapshot": {
    "snapshot": "snapshot_20260404_112031",
    "uuid": "MD9ReZ9ISwiegBVTzxQIqQ",
    "version_id": 6070099,
    "version": "6.7.0",
    "indices": [
      "logs"
    ],
    "include_global_state": true,
    "state": "SUCCESS",
    "start_time": "2026-04-04T05:50:31.916Z",
    "start_time_in_millis": 1775281831916,
    "end_time": "2026-04-04T05:50:32.331Z",
    "end_time_in_millis": 1775281832331,
    "duration_in_millis": 415,
    "failures": [],
    "shards": {
      "total": 1,
      "failed": 0,
      "successful": 1
    }
  }
}


## Restore Snapshot to Destination Cluster

This cell restores the snapshot from the source cluster to the destination cluster:

**Restore Configuration:**
- `indices: "*"`: Restore all indices from the snapshot
- `rename_pattern`: Pattern to match index names for renaming
- `rename_replacement`: How to rename indices (adds "restored_" prefix)

**Process:**
1. Constructs restore URL for the destination cluster
2. Sends POST request with restore payload
3. OpenSearch restores indices with new names to avoid conflicts
4. Prints restore operation details and status

**Purpose**: Demonstrates disaster recovery by restoring data from one cluster to another using S3 snapshots.

In [99]:


payload = {
    "indices": "*", 
    "rename_pattern": "(.+)",
    "rename_replacement": "restored_$1"
}

url = f"https://{destination_host}/_snapshot/{repository_name}/{snapshot_name}/_restore"

headers = {"Content-Type": "application/json"}

print(f"Starting restore of {snapshot_name} to {destination_host}...")

# 5. Execute POST request
try:
    response = requests.post(url, auth=auth_north, json=payload, headers=headers)
    response.raise_for_status()
    print("Restore initiated successfully:")
    print(json.dumps(response.json(), indent=2))
except requests.exceptions.HTTPError as err:
    print(f"Error: {err.response.status_code}")
    print(err.response.text)


Starting restore of snapshot_20260404_112031 to search-poc-no-ccr-1-dr-vrwia7ann6gpd3yrstznjlnnqu.us-east-1.es.amazonaws.com...
Restore initiated successfully:
{
  "accepted": true
}


## View Restored Data on Destination Cluster

This final cell queries the destination OpenSearch cluster to verify the restored data:

- Searches across all indices (`*`) on the destination cluster
- Uses `auth_north` for authentication with the destination cluster
- Extracts and displays document data in a formatted table
- Shows that the data has been successfully restored from the snapshot

**Expected Result**: Should display the same documents that were originally in the source cluster, now restored with "restored_" prefix on index names.

**Purpose**: Confirms that the backup and restore process worked correctly, demonstrating cross-cluster disaster recovery capabilities.

In [100]:

endpoint = os.getenv('DESTINATION_ENDPOINT')
url = f"https://{endpoint}/*/_search"
params = {"size": 10}

try:
    # 1. Execute the signed request
    response = requests.get(url, auth=auth_north, params=params)
    response.raise_for_status()
    
    # 2. Parse the JSON response
    results = response.json()
    hits = results.get('hits', {}).get('hits', [])

    # 3. Extract the data for the table
    table_data = []
    for hit in hits:
        source = hit.get('_source', {})
        table_data.append([
            hit.get('_id'),
            source.get('timestamp'),
            source.get('level'),
            source.get('message'),
            source.get('region')
        ])

    # 4. Display as a table
    headers = ["ID", "Timestamp", "Level", "Message", "Region"]
    print(tabulate(table_data, headers=headers, tablefmt="grid"))

except Exception as e:
    print(f"Error: {e}")


+------+---------------------------+---------+---------------------------------------+-----------+
|   ID | Timestamp                 | Level   | Message                               | Region    |
+======+===========================+=========+=======================================+===========+
|    1 | 2026-04-04T05:47:21+00:00 | INFO    | US-East Primary cluster populated     | us-east-1 |
+------+---------------------------+---------+---------------------------------------+-----------+
|    2 | 2026-04-04T05:47:21+00:00 | INFO    | Source data ready for CCR to AP-South | us-east-1 |
+------+---------------------------+---------+---------------------------------------+-----------+


# Demo 2: Cross-Cluster Replication (CCR) for Opensearch Version above 7.10 !

## Overview

Demo 2 demonstrates **AWS OpenSearch Cross-Cluster Replication**, a real-time data synchronization mechanism between two OpenSearch domains across different AWS regions. This enables disaster recovery, high availability, and data locality.

#### Documentation: [link](https://docs.aws.amazon.com/opensearch-service/latest/developerguide/replication.html)

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                         LEADER CLUSTER                          │
│              (ap-south-1: Mumbai Region)                         │
│                                                                  │
│  Endpoint: search-poc-no-ccr--new-1-a7rvkwro...               │
│  Index: sample-data-index3                                      │
│  Role: Leader replication role (read permissions)              │
│                                                                  │
│  [Write Documents] → [Maintain Index]                          │
└─────────────────────┬───────────────────────────────────────────┘
                      │
                      │ Real-Time Replication
                      │ Connection Alias: "connect-demo"
                      │
┌─────────────────────▼───────────────────────────────────────────┐
│                       FOLLOWER CLUSTER                          │
│              (us-east-1: N. Virginia Region)                     │
│                                                                  │
│  Endpoint: search-poc-no-ccr--new-1-dr-hmjlkk...              │
│  Index: sample-data-index3 (replicated)                         │
│  Role: Follower replication role (write permissions)           │
│                                                                  │
│  [Receive Updates] → [Keep in Sync]                            │
└──────────────────────────────────────────────────────────────────┘
```




## Step 1 & 2: Configure Cluster Endpoints and Populate Leader

This cell sets up the cross-cluster replication configuration and populates the leader cluster with sample data:

**Configuration Variables:**
- `LEADER_HOST`: Endpoint URL for the leader cluster (ap-south-1, Mumbai)
- `FOLLOWER_HOST`: Endpoint URL for the follower cluster (us-east-1, N. Virginia)
- `LEADER_ALIAS`: Connection alias name used in the console ("connect-demo")
- `INDEX_NAME`: Name of the index for replication ("sample-data-index6")

**Process:**
1. Creates a new index on the leader cluster
2. Adds sample documents with user actions and timestamps
3. Each document contains: user, action, and timestamp fields
4. Prints HTTP status codes for verification

**Data Added:**
- Document 1: User "Suraj" with login action
- Document 2: User "DevOps-Team" with deploy action

**Purpose**: Establishes the initial data that will be replicated in real-time to the follower cluster.

In [ ]:

LEADER_HOST = 'https://search-poc-no-ccr--new-1-a7rvkwro33rgaljyve44lnstnq.ap-south-1.es.amazonaws.com'
FOLLOWER_HOST = 'https://search-poc-no-ccr--new-1-dr-hmjlkk35wucrw5qx5ffbufxpqu.us-east-1.es.amazonaws.com'

LEADER_REGION = 'ap-south-1'
FOLLOWER_REGION = 'us-east-1'
LEADER_ALIAS = 'connect-demo' # The name you gave the connection in the console
INDEX_NAME = 'sample-data-index6'



leader_auth = auth_south
follower_auth = auth_north

headers = {"Content-Type": "application/json"}

# --- 2. Populate Leader Cluster (Mumbai) ---
print(f"[*] Creating index '{INDEX_NAME}' on Leader...")
# Creating the index
create_idx_res = requests.put(f"{LEADER_HOST}/{INDEX_NAME}", auth=leader_auth, headers=headers)
print(f"Status: {create_idx_res.status_code}, Response: {create_idx_res.text}")

# Adding sample data
print("[*] Adding sample documents to Leader...")
sample_docs = [
    {"user": "Suraj", "action": "login", "timestamp": "2026-04-03T10:00:00Z"},
    {"user": "DevOps-Team", "action": "deploy", "timestamp": "2026-04-03T10:05:00Z"}
]

for i, doc in enumerate(sample_docs):
    res = requests.post(f"{LEADER_HOST}/{INDEX_NAME}/_doc/{i}", auth=leader_auth, json=doc, headers=headers)
    print(f"Doc {i} status: {res.status_code}")



In [102]:
# --- 3. Register Remote Cluster & Initiate Replication on Follower (N. Virginia) ---
print(f"[*] Registering remote cluster on Follower...")

# First, register the leader as a remote cluster on the follower
remote_cluster_payload = {
    "persistent": {
        "cluster.remote": {
            f"{LEADER_ALIAS}.seeds": f"{LEADER_HOST.replace('https://', '').split('/')[0]}:9300",
            f"{LEADER_ALIAS}.transport.compress": "true"
        }
    }
}

settings_url = f"{FOLLOWER_HOST}/_cluster/settings"
settings_res = requests.put(settings_url, auth=follower_auth, json=remote_cluster_payload, headers=headers)
print(f"Remote cluster registration status: {settings_res.status_code}")

# Now start the replication
print(f"\n[*] Starting replication for '{INDEX_NAME}' on Follower...")

# Replication requires roles to be specified
replication_payload = {
   "leader_alias": LEADER_ALIAS,
   "leader_index": INDEX_NAME,
   "use_roles": {
      "leader_cluster_role": "all_access",
      "follower_cluster_role": "all_access"
   }
}

# The replication start API is called on the FOLLOWER endpoint
start_url = f"{FOLLOWER_HOST}/_plugins/_replication/{INDEX_NAME}/_start"

repl_res = requests.put(start_url, auth=follower_auth, json=replication_payload, headers=headers)

if repl_res.status_code == 200:
    print("[SUCCESS] Replication initiated successfully.")
    print(json.dumps(repl_res.json(), indent=2))
else:
    print(f"[ERROR] Failed to start replication: {repl_res.status_code}")
    print(repl_res.text)

[*] Registering remote cluster on Follower...
Remote cluster registration status: 401

[*] Starting replication for 'sample-data-index6' on Follower...
[SUCCESS] Replication initiated successfully.
{
  "acknowledged": true
}


## Step 4: Check Replication Status

This cell queries the replication status on the follower cluster:

- Constructs the status URL using the replication API endpoint
- Makes authenticated GET request to retrieve replication status
- Returns replication state (RUNNING, PAUSED, etc.)
- Displays detailed status information including shard replication progress

**Purpose**: Monitor and verify that the cross-cluster replication is actively synchronizing data from the leader to the follower cluster.

In [103]:
# --- 4. Check Status ---
print("[*] Checking replication status...")
status_res = requests.get(f"{FOLLOWER_HOST}/_plugins/_replication/{INDEX_NAME}/_status", auth=follower_auth)
print(json.dumps(status_res.json(), indent=2))

[*] Checking replication status...
{
  "status": "SYNCING",
  "reason": "User initiated",
  "leader_alias": "connect-demo",
  "leader_index": "sample-data-index6",
  "follower_index": "sample-data-index6",
  "syncing_details": {
    "leader_checkpoint": -3,
    "follower_checkpoint": -3,
    "seq_no": 2
  }
}


## Step 5: Define Function to View Replicated Data

This cell defines a reusable function to query and display replicated documents from the follower cluster:

**Function Purpose**: Retrieves documents from the follower cluster and displays them in a formatted table for verification.

**Function Parameters:**
- `host`: Follower cluster endpoint URL
- `index`: Index name to query
- `auth`: AWS4Auth object for authentication
- `limit`: Number of documents to retrieve (default: 10)

**Process:**
1. Constructs search URL for the specified index
2. Executes authenticated GET request to retrieve documents
3. Extracts document ID, user, action, and timestamp from search hits
4. Formats results in a grid table using `tabulate`
5. Handles errors gracefully with exception handling

**Output**: Displays table with columns: ID, User, Action, Timestamp

In [104]:
def view_replicated_data(host, index, auth, limit=10):
    """
    Display replicated documents from the follower cluster in a formatted table.
    
    Args:
        host: Follower cluster endpoint URL
        index: Index name to query
        auth: AWS4Auth object for the follower cluster
        limit: Number of documents to retrieve (default: 10)
    """
    url = f"{host}/{index}/_search"
    params = {"size": limit}
    
    try:
        response = requests.get(url, auth=auth, params=params)
        response.raise_for_status()
        
        results = response.json()
        hits = results.get('hits', {}).get('hits', [])
        
        if not hits:
            print(f"[*] No documents found in index '{index}' on follower cluster")
            return
        
        # Extract data for table
        table_data = []
        for hit in hits:
            source = hit.get('_source', {})
            table_data.append([
                hit.get('_id'),
                source.get('user', 'N/A'),
                source.get('action', 'N/A'),
                source.get('timestamp', 'N/A')
            ])
        
        # Display results
        print(f"\n[SUCCESS] Retrieved {len(hits)} document(s) from follower cluster:")
        headers = ["ID", "User", "Action", "Timestamp"]
        print(tabulate(table_data, headers=headers, tablefmt="grid"))
        
    except Exception as e:
        print(f"[ERROR] Failed to retrieve replicated data: {e}")

# Call the function to view replicated data
view_replicated_data(FOLLOWER_HOST, INDEX_NAME, follower_auth)


[SUCCESS] Retrieved 2 document(s) from follower cluster:
+------+-------------+----------+----------------------+
|   ID | User        | Action   | Timestamp            |
+======+=============+==========+======================+
|    0 | Suraj       | login    | 2026-04-03T10:00:00Z |
+------+-------------+----------+----------------------+
|    1 | DevOps-Team | deploy   | 2026-04-03T10:05:00Z |
+------+-------------+----------+----------------------+
